# GDRN — Kiểm thử tải trên Kaggle CPU

Chạy ứng dụng, PostGIS và k6 trên cùng máy CPU, không cần Docker hoặc máy chủ bên ngoài.

1. Tại thư mục gốc: `python benchmark/package_cpu_source.py`.
2. Tải `benchmark/results/gdrn-cpu-source.zip` lên **Dataset Kaggle riêng tư**, rồi chọn **Add Input** trong notebook.
3. Chọn **Accelerator: None**, **Internet: On**, rồi **Run All**. Công cụ tự tạo thông tin xác thực dùng một lần.
4. Lấy `gdrn-cpu-results.zip` ở **Output**. Chỉ xác nhận hoàn tất khi `manifest.json` có `status: completed` và đủ phép đo.

Mặc định: 4 kịch bản × 4 mức tải (1/10/50/100 người dùng ảo) × 3 lần; mỗi lần 10 giây làm nóng, 30 giây giữ tải, 10 giây giảm tải. Dừng khi k6 lỗi hoặc tỷ lệ lỗi >=1%. Máy dùng chung bộ phát tải, ứng dụng và cơ sở dữ liệu; không so trực tiếp với máy chủ độc lập.


In [ ]:
from pathlib import Path
import json
import subprocess
import zipfile
import shutil

inputs = Path('/kaggle/input')
archives = list(inputs.rglob('gdrn-cpu-source.zip'))
# Kaggle may unpack the uploaded ZIP when creating a Dataset.
source_roots = [p.parent for p in inputs.rglob('pom.xml')
                if (p.parent / 'benchmark/run_cpu.py').is_file()]
ROOT = Path('/kaggle/working/gdrn-cpu-source')
ROOT.mkdir(exist_ok=True)
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            target = (ROOT / name).resolve()
            if not target.is_relative_to(ROOT.resolve()):
                raise RuntimeError('Invalid archive path')
        archive.extractall(ROOT)
elif not archives and len(source_roots) == 1:
    shutil.copytree(source_roots[0], ROOT, dirs_exist_ok=True)
else:
    raise RuntimeError('Add exactly one GDRN source Dataset (ZIP or extracted files).')
print('Source ready:', ROOT)


In [ ]:
# Requires internet for Ubuntu packages, pinned k6 and Maven dependencies.
subprocess.run(['bash', str(ROOT / 'benchmark/setup-cpu.sh')], check=True)


## Chạy và lưu bằng chứng

Mỗi lần lặp dùng cơ sở dữ liệu đã đặt lại; kịch bản đọc/hỗn hợp tạo 1.000 báo cáo với hạt giống 20261024. Kịch bản `write` bắt đầu từ cơ sở dữ liệu rỗng.
`resources.csv` ghi CPU/RAM máy và RSS ứng dụng/k6/Postgres mỗi giây; tổng RSS Postgres có thể tính trùng trang bộ nhớ dùng chung.
Bảng tính yêu cầu nghiệp vụ, loại đăng nhập, gồm cả làm nóng/giảm tải; không phải số liệu riêng giai đoạn giữ tải.


In [ ]:
OUTPUT = Path('/kaggle/working/gdrn-cpu-results')
# Reduce repetitions/duration for smoke only; manifest records the actual configuration.
command = ['python3', str(ROOT / 'benchmark/run_cpu.py'), '--output', str(OUTPUT),
           '--vus', '1', '10', '50', '100', '--repetitions', '3',
           '--dataset-size', '1000', '--duration', '30', '--warmup', '10', '--ramp-down', '10']
try:
    subprocess.run(command, cwd=ROOT, check=True)
finally:
    if OUTPUT.exists():
        shutil.make_archive('/kaggle/working/gdrn-cpu-results', 'zip', OUTPUT)
        print('Evidence archive: /kaggle/working/gdrn-cpu-results.zip')


In [ ]:
from IPython.display import Markdown, display, FileLink
runs = sorted(OUTPUT.glob('*/manifest.json'))
if not runs:
    raise RuntimeError('No benchmark manifest found.')
latest = runs[-1].parent
manifest = json.loads((latest / 'manifest.json').read_text())
print('Environment:', manifest['environment'], '| Status:', manifest['status'])
display(Markdown((latest / 'REPORT.md').read_text()))
display(FileLink('/kaggle/working/gdrn-cpu-results.zip'))
